In [4]:
import pandas as pd

files = {
    'results': '../data/raw/archive/results.csv',
    'fixtures': '../data/raw/archive_2/wc_2026_fixtures.csv',
    'teams':    '../data/raw/archive_2/wc_2026_teams.csv',
    'elo':      '../data/raw/archive_3/elo_ratings_wc2026.csv',
    'train':    '../data/raw/archive_4/train.csv',
}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

for name, path in files.items():
    df = pd.read_csv(path)
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))


results: (49477, 9)
Columns: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England         0.0         0.0   Friendly  Glasgow   
1  1873-03-08   England  Scotland         4.0         2.0   Friendly   London   
2  1874-03-07  Scotland   England         2.0         1.0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  

fixtures: (104, 15)
Columns: ['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date', 'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach', 'team2_confederation', 'team2_fifa_rank', 'team2_coach']
  group        stage        team1         team2           venue         city  \
0     A  Group Stage       Mexico  South Africa  Estadio Azteca  Mexico City   
1     A  Group Stage  South Korea       Czechia   Estad

In [6]:
results  = pd.read_csv('../data/raw/archive/results.csv')
fixtures = pd.read_csv('../data/raw/archive_2/wc_2026_fixtures.csv')
teams    = pd.read_csv('../data/raw/archive_2/wc_2026_teams.csv')
elo      = pd.read_csv('../data/raw/archive_3/elo_ratings_wc2026.csv')
train    = pd.read_csv('../data/raw/archive_4/train.csv')

In [7]:
# Check team name consistency between datasets
results_teams = set(results['home_team'].unique()) | set(results['away_team'].unique())
fixtures_teams = set(fixtures['team1'].unique()) | set(fixtures['team2'].unique())

# Teams in fixtures not found in results
missing = fixtures_teams - results_teams
print(f"Teams in fixtures not in results: {len(missing)}")
print(sorted(missing))

Teams in fixtures not in results: 66
['1A', '1B', '1C', '1D', '1E', '1F', '1G', '1H', '1I', '1J', '1K', '1L', '2A', '2B', '2C', '2D', '2E', '2F', '2G', '2H', '2I', '2J', '2K', '2L', '3rd Place', 'Best 3rd #1', 'Best 3rd #2', 'Best 3rd #3', 'Best 3rd #4', 'Best 3rd #5', 'Best 3rd #6', 'Best 3rd #7', 'Best 3rd #8', 'Czechia', 'Finalist 1', 'Finalist 2', 'QF1', 'QF2', 'QF3', 'QF4', 'QF5', 'QF6', 'QF7', 'QF8', 'R32 W1', 'R32 W10', 'R32 W11', 'R32 W12', 'R32 W13', 'R32 W14', 'R32 W15', 'R32 W16', 'R32 W2', 'R32 W3', 'R32 W4', 'R32 W5', 'R32 W6', 'R32 W7', 'R32 W8', 'R32 W9', 'SF1', 'SF2', 'SF3', 'SF4', 'Türkiye', 'USA']


In [8]:
# Verify the three real mismatches
for name in ['Czech Republic', 'Turkey', 'United States']:
    found = name in results_teams
    print(f"{name}: {'found' if found else 'NOT found'}")

# Fix name mapping
name_map = {
    'Czechia': 'Czech Republic',
    'Türkiye': 'Turkey',
    'USA':     'United States'
}

fixtures['team1'] = fixtures['team1'].replace(name_map)
fixtures['team2'] = fixtures['team2'].replace(name_map)

# Keep only group stage for now
group_stage = fixtures[fixtures['stage'] == 'Group Stage'].copy()
print(f"\nGroup stage matches: {len(group_stage)}")

Czech Republic: found
Turkey: found
United States: found

Group stage matches: 72


In [9]:
# Check tournament types and counts
print(results['tournament'].value_counts().head(20))

# Check date range
print(f"\nDate range: {results['date'].min()} → {results['date'].max()}")

# Check missing values
print(f"\nMissing values:\n{results.isnull().sum()}")

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
Name: count, dtype: int64

Date range: 1872-11-30 → 2026-06-27

Missi

In [10]:
# Keep only competitive matches
competitive = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa América',
    'African Cup of Nations',
    'AFC Asian Cup',
    'Gold Cup',
    'UEFA Nations League',
    'CONCACAF Nations League',
]

df = results[results['tournament'].isin(competitive)].copy()
print(f"Competitive matches: {len(df)}")

# Target variable distribution
df['result'] = df.apply(
    lambda r: 'H' if r['home_score'] > r['away_score']
    else ('A' if r['away_score'] > r['home_score'] else 'D'),
    axis=1
)
print(f"\nOutcome distribution:\n{df['result'].value_counts(normalize=True).round(3)}")

# World Cup matches only
wc = df[df['tournament'] == 'FIFA World Cup']
print(f"\nWorld Cup matches only:\n{wc['result'].value_counts(normalize=True).round(3)}")

Competitive matches: 16654

Outcome distribution:
result
H    0.490
A    0.292
D    0.218
Name: proportion, dtype: float64

World Cup matches only:
result
H    0.438
A    0.303
D    0.259
Name: proportion, dtype: float64


In [11]:
# Check neutral flag distribution
print("Neutral flag in competitive matches:")
print(df['neutral'].value_counts())

# Outcome distribution on neutral ground only
neutral_df = df[df['neutral'] == True]
print(f"\nOutcome on neutral ground ({len(neutral_df)} matches):")
print(neutral_df['result'].value_counts(normalize=True).round(3))

# Outcome distribution in WC specifically
print(f"\nNeutral flag in WC matches:")
print(wc['neutral'].value_counts())

Neutral flag in competitive matches:
neutral
False    12600
True      4054
Name: count, dtype: int64

Outcome on neutral ground (4054 matches):
result
H    0.417
A    0.341
D    0.241
Name: proportion, dtype: float64

Neutral flag in WC matches:
neutral
True     906
False    130
Name: count, dtype: int64


### Feature Engineering ###

In [12]:
import numpy as np

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

def get_recent_form(df, team, date, n=10):
    """Get win rate and avg goals for a team in last N matches before date"""
    mask = (
        ((df['home_team'] == team) | (df['away_team'] == team)) &
        (df['date'] < date)
    )
    recent = df[mask].tail(n)
    if len(recent) == 0:
        return pd.Series({'win_rate': 0.5, 'avg_goals_scored': 1.5, 'avg_goals_conceded': 1.5})

    wins, goals_for, goals_against = 0, 0, 0
    for _, row in recent.iterrows():
        if row['home_team'] == team:
            goals_for += row['home_score']
            goals_against += row['away_score']
            if row['result'] == 'H': wins += 1
        else:
            goals_for += row['away_score']
            goals_against += row['home_score']
            if row['result'] == 'A': wins += 1

    return pd.Series({
        'win_rate': wins / len(recent),
        'avg_goals_scored': goals_for / len(recent),
        'avg_goals_conceded': goals_against / len(recent)
    })

# Test on one match
test_date = pd.Timestamp('2022-11-20')
print(get_recent_form(df, 'France', test_date, n=10))

win_rate              0.5
avg_goals_scored      2.0
avg_goals_conceded    1.0
dtype: float64


In [13]:
# Prepare Elo - take last snapshot per year per country
elo['snapshot_date'] = pd.to_datetime(elo['snapshot_date'])
elo_latest = elo.sort_values('snapshot_date').groupby(['country', 'year']).last().reset_index()

def get_elo(team, date, elo_df):
    """Get most recent Elo rating for a team before given date"""
    year = date.year
    mask = (elo_df['country'] == team) & (elo_df['year'] < year)
    recent = elo_df[mask]
    if len(recent) == 0:
        return 1500  # default rating
    return recent.sort_values('year').iloc[-1]['rating']

# Test
print(get_elo('France', pd.Timestamp('2022-11-20'), elo_latest))
print(get_elo('Brazil', pd.Timestamp('2022-11-20'), elo_latest))
print(get_elo('Argentina', pd.Timestamp('2022-11-20'), elo_latest))


2114
2149
2101


### Building final dataset ###


In [15]:
from tqdm import tqdm

rows = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    date = row['date']
    home = row['home_team']
    away = row['away_team']

    # Home team form
    home_form = get_recent_form(df, home, date, n=10)
    # Away team form
    away_form = get_recent_form(df, away, date, n=10)

    # Elo ratings
    home_elo = get_elo(home, date, elo_latest)
    away_elo = get_elo(away, date, elo_latest)

    rows.append({
        'date':                    date,
        'home_team':               home,
        'away_team':               away,
        'neutral':                 row['neutral'],
        'tournament':              row['tournament'],
        'home_win_rate':           home_form['win_rate'],
        'home_avg_goals_scored':   home_form['avg_goals_scored'],
        'home_avg_goals_conceded': home_form['avg_goals_conceded'],
        'away_win_rate':           away_form['win_rate'],
        'away_avg_goals_scored':   away_form['avg_goals_scored'],
        'away_avg_goals_conceded': away_form['avg_goals_conceded'],
        'elo_diff':                home_elo - away_elo,
        'home_elo':                home_elo,
        'away_elo':                away_elo,
        'result':                  row['result'],
    })

features_df = pd.DataFrame(rows)
print(f"Dataset shape: {features_df.shape}")
print(features_df.head(3))

100%|██████████| 16654/16654 [03:09<00:00, 88.10it/s]


Dataset shape: (16654, 15)
        date  home_team away_team  neutral    tournament  home_win_rate  \
0 1916-07-02      Chile   Uruguay     True  Copa América            0.5   
1 1916-07-06  Argentina     Chile    False  Copa América            0.5   
2 1916-07-08     Brazil     Chile     True  Copa América            0.5   

   home_avg_goals_scored  home_avg_goals_conceded  away_win_rate  \
0                    1.5                      1.5            0.5   
1                    1.5                      1.5            0.0   
2                    1.5                      1.5            0.0   

   away_avg_goals_scored  away_avg_goals_conceded  elo_diff  home_elo  \
0                    1.5                      1.5      -413      1500   
1                    0.0                      4.0       440      1940   
2                    0.5                      5.0       410      1910   

   away_elo result  
0      1913      A  
1      1500      H  
2      1500      D  
